# 🔧 Data Processing

**✍️ Author:** Hayriye Anıl  
**📘 Blog Series:** Time Series Analysis & Forecasting  

## 🎯 Purpose

This notebook merges weather datasets from separate years into a single dataset and performs essential data processing steps, including handling missing values and converting data types into their correct formats.

## 📂 Contents

- 📥 Data loading and initial inspection  
- ⏱️ Timestamp handling and resampling  
- 🧩 Missing data analysis and imputation & checking duplicate values
- 📊 Example visualizations/code snippets for the first article (Understanding Time Series Data)

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

### Data loading and initial inspection

In [ ]:
RAW_DATA_DIR = Path().resolve().parent / "data" / "raw_data"
PROCESSED_DATA_DIR = Path().resolve().parent / "data" / "processed_data"

In [ ]:
files = list(RAW_DATA_DIR.glob('*.csv'))
files

In [ ]:
data_frame = []
for file in files:
    data = pd.read_csv(file, sep=',', skiprows=3)
    data_frame.append(data)
    
dataset = pd.concat(data_frame, axis=0)
dataset

In [ ]:
dataset.info(verbose=True, show_counts=True)

In [ ]:
dataset.head(5)

### Timestamp handling and resampling  

In [ ]:
dataset['time'] = pd.to_datetime(dataset['time'], format='%Y-%m-%dT%H:%M')

In [ ]:
dataset.set_index('time', inplace=True)

In [ ]:
dataset

### Missing data analysis and imputation  

In [ ]:
dataset.isnull().sum()

In [ ]:
# Percantage of missing values in the dataset
dataset.isnull().mean() * 100

In [ ]:
dataset[dataset["wind_speed_180m (km/h)"].isnull()][["wind_speed_180m (km/h)", "wind_direction_180m (°)", "soil_temperature_0cm (°C)", "soil_temperature_6cm (°C)", "soil_temperature_18cm (°C)", "soil_temperature_54cm (°C)" ]]

In [ ]:
# Remove "precipitation_probability (%)" and "weather_code (wmo code)" columns
dataset.drop(["precipitation_probability (%)", "weather_code (wmo code)"], axis=1, inplace=True)

In [ ]:
# Start dataset from December 2022
dataset = dataset[(dataset.index >= "2022-12-01") & (dataset.index < "2026-01-01")]
dataset

### Check duplicate values

In [ ]:
dataset[dataset.duplicated()]

In [ ]:
dataset[dataset.index == "2023-01-01"]

In [ ]:
dataset.drop_duplicates(inplace=True)

In [ ]:
dataset.head(5)

In [ ]:
dataset[dataset.duplicated()]

In [ ]:
dataset[dataset.index == "2023-01-01"]

In [ ]:
dataset.info(verbose=True, show_counts=True)

In [ ]:
dataset.isnull().sum()

In [ ]:
dataset.to_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", index=True, header=True)

### Example visualizations/code snippets for the first article (Understanding Time Series Data)

In [ ]:
dataset = pd.read_csv(f"{PROCESSED_DATA_DIR}/processed_dataset.csv", index_col=0, parse_dates=True)
dataset

#### Time Series Data

In [ ]:
fig = px.line(dataset, x=dataset.index, y='temperature_2m (°C)', title='Istanbul Temperature Over Time')
fig.update_layout(xaxis_title='Time', yaxis_title='Temperature (°C)')
fig.show()

#### Cross-Section Data

In [ ]:
students = ['Alice', 'Bob', 'Charlie', 'David', 'Eve', 'Frank']
exam_scores = [85, 92, 78, 88, 94, 79]

fig = go.Figure(data=[go.Bar(x=students, y=exam_scores)])

fig.update_layout(
    title="University Entrance Exam Scores in a Given Year",
    xaxis_title="Students",
    yaxis_title="Exam Scores",
)
fig.show()

#### Understand Timestamp: Time zone and Time Scale

In [ ]:
import pytz
from datetime import datetime

local_time = datetime(2025, 3, 26, 12, 0, 0) 
local_tz = pytz.timezone('Europe/Istanbul')

localized_time = local_tz.localize(local_time)

utc_time = localized_time.astimezone(pytz.utc)

print("Local time:", localized_time)
print("UTC time:", utc_time)


#### Resampling

In [ ]:
daily_temp = dataset.resample('D').agg({
    'apparent_temperature (°C)': 'mean',
})
daily_temp

In [ ]:
daily_temp.reset_index(inplace=True)
hourly_dates = pd.date_range(start=daily_temp['time'].min(), end=daily_temp['time'].max(), freq='h')
df_hourly = daily_temp.set_index('time').reindex(hourly_dates, method=None)
df_hourly['apparent_temperature_interpolated'] = df_hourly['apparent_temperature (°C)'].interpolate(method='linear')
df_hourly